# **PYSPARK INTERVIEW QUESTIONS - ANSH LAMBA**

In [0]:
csv_path = 'abfss://dbricks-sales@pysparkpractice.dfs.core.windows.net/BigMart Sales.csv'

df_csv = spark.read.format('csv')\
            .option('inferSchema', True)\
            .option('header',True)\
            .load(csv_path)

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

**Q1 While ingesting customer data from an external source, you notice duplicate entries. How would you remove duplicates and retain only the latest entry based on a timestamp column?**

In [0]:
data = [("101", "2023-12-01", 100), ("101", "2023-12-02", 150), 
        ("102", "2023-12-01", 200), ("102", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

**Solution**

In [0]:
df = df.withColumn('date',col('date').cast(DateType()))

df.sort(desc(col('date'))).dropDuplicates(['product_id']).display()

**2. While processing data from multiple files with inconsistent schemas, you need to merge them into a single DataFrame. How would you handle this inconsistency in PySpark?**

**Solution**

In [0]:
 df = spark.read.format('parquet')\
        .option('mergetSchema',True)\
        .option('FileStore/Datafiles')

**4. You are working with a real-time data pipeline, and you notice missing values in your streaming data Column - Category. How would you handle null or missing values in such a scenario?**

**df_stream = spark.readStream.schema("id INT, value STRING").csv("path/to/stream")**

In [0]:
# df_csv.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))\
#     .fillna('NA Item_Weight',subset=['Item_Weight', 'Outlet_Size']).display()


df_csv.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))\
    .fillna({'Item_Weight': 'NA Item_Weight','Outlet_Size':'NA Outlet_Size'},subset=['Item_Weight','Outlet_Size']).display()

**5. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this information?**

In [0]:
data = [("user1", 5), ("user2", 8), ("user3", 2), ("user4", 10), ("user2", 3)]
columns = ["user_id", "actions"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.groupBy('user_id').agg(sum(col('actions')).alias('Total Actions')).sort(col('Total Actions'), ascending = False).limit(5).display()

**6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?**

In [0]:
data = [("cust1", "2023-12-01", 100), ("cust2", "2023-12-02", 150),
        ("cust1", "2023-12-03", 200), ("cust2", "2023-12-04", 250)]
columns = ["customer_id", "transaction_date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.withColumn('transaction_date', col('transaction_date').cast(DateType()))
window_spec = Window.partitionBy('customer_id').orderBy(col('transaction_date').desc())

df.withColumn('dense', dense_rank().over(window_spec)).filter(col('dense') == 1).drop('dense').display()

**7. You need to identify customers who haven’t made any purchases in the last 30 days. How would you filter such customers?**

In [0]:
data = [("cust1", "2025-12-01"), ("cust2", "2024-11-20"), ("cust3", "2024-11-25")]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df.filter(datediff(current_date(), 'last_purchase_date')  )

**8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?**

In [0]:
data = [("customer1", "The product is great"), ("customer2", "Great product, fast delivery"), ("customer3", "Not bad, could be better")]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df.withColumn('feedback', split(col('feedback'),' '))\
    .withColumn('feedback', explode(col('feedback')))\
    .groupBy(lower(col('feedback')))\
    .agg(count(col('feedback')).alias('word count')).sort('word count', ascending = False).display()

**9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-02", 200),
        ("product1", "2023-12-03", 150), ("product2", "2023-12-04", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy('product_id').orderBy('date').rowsBetween(Window.unboundedPreceding, Window.currentRow)


df.withColumn('Cumulative Sum', sum('sales').over(window_spec)).display()

**10. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?**

In [0]:
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.window import Window

ordered_df = df.withColumn("original_order", monotonically_increasing_id())
window_spec = Window.partitionBy("name", "age").orderBy("original_order")


ordered_df.withColumn('row_num',row_number().over(window_spec))\
    .filter(col('row_num')==1)\
    .sort('original_order')\
    .drop('original_order','row_num').display()

**11. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?**

In [0]:
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60), 
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df.groupBy('user_id').agg(avg('duration').alias('avg_duration')).display()

**12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150), 
        ("product1", "2023-12-02", 200), ("product2", "2023-12-02", 250),
        ("product3", "2023-11-02", 200), ("product4", "2023-11-02", 250),
        ("product3", "2023-11-02", 200), ("product4", "2023-11-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
# window_spec = Window.partitionBy('date').orderBy(col('sales').desc())

# df.withColumn('row_num', row_number().over(window_spec))\
#     .filter(col('row_num') == 1)\
#     .display()
from pyspark.sql.window import Window
df_new = df.withColumn('date', to_date('date'))

df_new.withColumn('date',month('date')).groupBy('date','product_id').agg(sum('sales').alias('sum_sales'))\
    .withColumn('ranking',dense_rank().over(Window.partitionBy('date').orderBy(col('sum_sales').desc())))\
    .filter(col('ranking') == 1).drop('ranking').display()

**13. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions, and sometimes updates can cause inconsistent reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?**

In [0]:
from delta.tables import DeltaTable

# The source path is a CSV file, so create a Delta version first and then load it as a Delta table.
# df_csv.write.format("delta").mode("overwrite").save(temp_delta_path)

# delta_tbl is a DeltaTable object representing the Delta Lake table at the specified path.
delta_tbl = DeltaTable.forPath(spark, csv_path)

delta_tbl.alias('src').merge(df_csv.alias('trg'),"src.id == trg.id")\
                        .whenNotMatchedInsertAll()\
                        .whenMatchedUpdateAll()\
                        .execute()

# Example: To ensure ACID compliance, you can perform updates, deletes, and merges using delta_tbl.
# For instance, to view the table's history:
display(delta_tbl.history())

# To update records:
# delta_tbl.update(condition, set)

# To delete records:
# delta_tbl.delete(condition)

# To merge data:
# delta_tbl.merge(source, condition, set)

**14. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (Almost). How would you do this?**

In [0]:
df.read.format('parquet').option('inferSchema',True).load(path)

**15. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?**

In [0]:
# Use DROPMALFORMED mode while reading csv file into Df

df = spark.read.format('csv')\
            .option("mode","DROPMALFORMED")\
            .load(csv_path)

**22. You have a dataset containing the names of employees and their departments. You need to find the department with the most employees.**

In [0]:
data = [("Alice", "HR"), ("Bob", "Finance"), ("Charlie", "HR"), ("David", "Engineering"), ("Eve", "Finance")]
columns = ["employee_name", "department"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
count_df = df.groupBy('department').agg(count(col('department')).alias('count'))

max_count = count_df.agg(max(col('count')).alias('max_count')).collect()[0][0]

count_df.filter(col('count') == max_count).display()

**23. While processing sales data, you need to classify each transaction as either 'High' or 'Low' based on its amount. How would you achieve this using a when condition**

In [0]:
data = [("product1", 100), ("product2", 300), ("product3", 50)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.withColumn('Category', when(col('sales') > 100, 'High')\
                        .otherwise('Low')).display()

**24. While analyzing a large dataset, you need to create a new column that holds a timestamp of when the record was processed. How would you implement this and what can be the best USE CASE?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.withColumn('load_ts', current_timestamp()).display()

**25. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it. How would you achieve this?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.createOrReplaceTempView('product')
spark.sql("select * from product limit 1").display()

**26. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it (FROM DIFFERENT NOTEBOOKS AS WELL)?**

In [0]:
df.createOrReplaceGlobalTempView('Global_view')


**27. You need to query data from a PySpark DataFrame using SQL, but the data includes a nested structure. How would you flatten the data for easier querying?**

In [0]:
data = [("product1", {"price": 100, "quantity": 2}), 
        ("product2", {"price": 200, "quantity": 3})]
columns = ["product_id", "product_info"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.select('product_id','product_info.price','product_info.quantity').display()

**28. You are ingesting data from an external API in JSON format where the schema is inconsistent. How would you handle this situation to ensure a robust pipeline?**

In [0]:
df = spark.read.format('json').option("mergeSchema",True).load(csv_path)

**29. While reading data from Parquet, you need to optimize performance by partitioning the data based on a column. How would you implement this?**

In [0]:
df.write.format("parquet").partitionBy("category").save("location")

df.read.format("parquet").partitionBy("category").load("location")

**30. You are working with a large dataset in Parquet format and need to ensure that the data is written in an optimized manner with proper compression. How would you accomplish this?**

In [0]:
df.write.format("parquet").option("compression","snappy")

**31. Your company uses a large-scale data pipeline that reads from Delta tables and processes data using complex aggregations. However, performance is becoming an issue due to the growing dataset size. How would you optimize the performance of the pipeline?**

In [0]:
%sql
OPTIMIZE mydeltatable ZORDER BY ('order_date')

**42. How does Delta Lake Time Travel Possible?**

In [0]:
%sql
DESCRIBE HISTORY TBL

RESTORE TBL TO VERSION AS OF 2

**43. You are processing sales data. Group by product categories and create a list of all product names in each category.**

In [0]:
data = [("Electronics", "Laptop"), ("Electronics", "Smartphone"), ("Furniture", "Chair"), ("Furniture", "Table")]
columns = ["category", "product"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.groupBy('category').agg(collect_list('product')).display()

**44. You are analyzing orders. Group by customer IDs and list all unique product IDs each customer purchased.**

In [0]:
data = [(101, "P001"), (101, "P002"), (102, "P001"), (101, "P001")]
columns = ["customer_id", "product_id"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df.groupBy('customer_id').agg(collect_set("product_id")).display()


**45. For customer records, combine first and last names only if the email address exists.**

In [0]:
data = [("John", "Doe", "john.doe@example.com"), ("Jane", "Smith", None)]
columns = ["first_name", "last_name", "email"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import when, concat
df.withColumn('full_name', when(col('email').isNotNull(), concat_ws('-',col('first_name'), col('last_name'))).otherwise(None)).display()


**46. You have a DataFrame containing customer IDs and a list of their purchased product IDs. Calculate the number of products each customer has purchased.**

In [0]:
data = [
    (1, ["prod1", "prod2", "prod3"]),
    (2, ["prod4"]),
    (3, ["prod5", "prod6"]),
]
myschema = "customer_id INT ,product_ids array<STRING>"

df = spark.createDataFrame(data, myschema)
df.display()

In [0]:
from pyspark.sql.functions import size, col, count_distinct
df.withColumn('Sales count', size(col('product_ids'))).display()

**47. You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes.**

In [0]:
data = [
    ("1",),
    ("123",),
    ("4567",),
]
schema = ["employee_id"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df.select(rpad('employee_id',6,'0').alias('padded')).display()

**48. You need to validate phone numbers by checking if they start with "91"**

In [0]:
data = [
    ("911234567890",),
    ("811234567890",),
    ("912345678901",),
]
schema = ["phone_number"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
# df.withColumn('Validation', when(col('phone_number').startswith('91'),lit('Valid')).otherwise(lit('Not Valid'))).display()


df.withColumn('Validation',when(substring(col('phone_number'),1,2) == '91',lit('Valid')).otherwise(lit('Not Valid'))).display()

**49. You have a dataset with courses taken by students. Calculate the average number of courses per student.**

In [0]:
data = [
    (1, ["Math", "Science"]),
    (2, ["History"]),
    (3, ["Art", "PE", "Biology"]),
]
schema = ["student_id", "courses"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df.withColumn('course_size', size(col('courses'))).groupBy().agg(avg('course_size')).display()

**50. You have a dataset with primary and secondary contact numbers. Use the primary number if available; otherwise, use the secondary number.**

In [0]:
data = [
    (None, "1234567890"),
    ("9876543210", None),
    ("7894561230", "4567891230"),
]
schema = ["primary_contact", "secondary_contact"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
# df.withColumn('Validation', when(col('primary_contact').isNull(), col('secondary_contact')).otherwise(col('primary_contact'))).display()


df.withColumn('Validation',coalesce(col('primary_contact'),col('secondary_contact'))).display()

**51. You are categorizing product codes based on their lengths. If the length is 5, label it as "Standard"; otherwise, label it as "Custom".**

In [0]:
data = [
    ("prod1",),
    ("prd234",),
    ("pr9876",),
]
schema = ["product_code"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df.withColumn('Category', when(length(col('product_code'))== 5, lit('Standard')).otherwise(lit('Not Standard'))).display()